# Bluesky: social posts - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 2, 11:00.** The social-media stream, honestly.

**What happened to the classics.** X/Twitter's free API was withdrawn in 2023 (search now
needs a paid tier); Facebook's CrowdTangle was shut down in 2024 (replaced by an
application-only research library). Neither is usable in class.

**Bluesky is open**, with one catch: full-text `searchPosts` needs authentication, but
**account discovery and public feeds do not**. So we monitor *accounts* rather than
search *keywords*: find public-health accounts, read their public posts, and look for
outbreak chatter. No key, no login.

> Captured example of what a coding agent (Codex, Claude Code, or Antigravity CLI)
> produces from the **Lane A** prompts.


## About this data source

**Bluesky.** A public social network with an *open door*: anyone can read public posts
without an account, which is exactly what Twitter stopped allowing. Think of it as:
*"a Twitter-shaped network you are still allowed to read."*

- **Explore it in a browser:** <https://bsky.app/> (search *dengue* or *outbreak* in the
  site's own search box, then open a public-health account and scroll its feed, which is
  exactly what this notebook reads)

**The catch:** keyword search through the API needs a login, but *finding accounts* and
*reading their public posts* do not. So we watch **accounts**, not keywords.

> **What happened to the classics:** X/Twitter went paid-API in 2023; Facebook's CrowdTangle
> shut down in 2024 (research access by application only). Our own COVID exercise file still
> has Twitter, Kinsa and Cuebiq columns, and all three of those streams are now closed.
> Novel data streams are fragile.


## Step 0: helpers (public AT Protocol, no auth)


In [ ]:
import pandas as pd, matplotlib.pyplot as plt, os, json, re, time
import urllib.request, urllib.parse

UA = {'User-Agent': 'SISMID2026-course/1.0 (your-email@example.com)'}

def cache_path(fname):
    for p in (f'../data/{fname}', f'data/{fname}', f'./{fname}'):
        if os.path.exists(p): return p
    return None

def get_json(url, timeout=45):
    return json.loads(urllib.request.urlopen(
        urllib.request.Request(url, headers=UA), timeout=timeout).read())

API = 'https://public.api.bsky.app/xrpc'

def bsky(method, **params):
    """Call a public Bluesky AppView endpoint. No authentication needed for
    searchActors / getAuthorFeed / resolveHandle. (searchPosts DOES need auth.)"""
    return get_json(f'{API}/{method}?' + urllib.parse.urlencode(params))


## Step 1: discover public-health accounts

`app.bsky.actor.searchActors` is open, so we can build the watch list programmatically.

**Prompt used (Lane A):**

> *Using the public Bluesky AppView (https://public.api.bsky.app/xrpc, no auth), call*
> *app.bsky.actor.searchActors for terms like 'epidemiology', 'public health',*
> *'outbreak', 'infectious disease'. Build a watch list of handles + display names.*


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
SEEDS = ['epidemiology', 'public health', 'outbreak', 'infectious disease',
         'virology', 'global health']
accounts = {}
try:
    for t in SEEDS:
        for a in bsky('app.bsky.actor.searchActors', q=t, limit=10).get('actors', []):
            accounts[a['handle']] = a.get('displayName') or ''
        time.sleep(0.4)
except Exception as e:
    p = cache_path('bluesky_health_accounts.csv')
    print('Live discovery failed:', e, '-> cache', p)
    accounts = dict(pd.read_csv(p).fillna('').values)
print(f'{len(accounts)} accounts on the watch list, e.g.:')
for h, d in list(accounts.items())[:6]: print(f'   @{h:36s} {d[:40]}')


## Step 2: read their public feeds and look for disease chatter

`app.bsky.feed.getAuthorFeed` is open too. We scan recent posts for disease keywords.

**Prompt used (Lane A):**

> *For each account call app.bsky.feed.getAuthorFeed (no auth) and pull recent posts.*
> *Flag posts mentioning disease keywords (dengue, influenza, covid, measles, rsv,*
> *outbreak, h5n1, malaria, mpox). Return a tidy table: date, handle, keywords, text.*
> *Use a WORD-BOUNDARY regex, not a substring test.*


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
KEYWORDS = ['dengue','influenza','covid','measles','rsv','outbreak','h5n1',
            'malaria','cholera','mpox','avian flu','bird flu','flu']

def scan(accounts, n_accounts=25, per_account=50):
    pat = re.compile(r'\b(' + '|'.join(re.escape(k) for k in KEYWORDS) + r')\b', re.I)
    rows, scanned = [], 0
    for h in list(accounts)[:n_accounts]:
        try:
            feed = bsky('app.bsky.feed.getAuthorFeed', actor=h, limit=per_account).get('feed', [])
        except Exception:
            continue
        for item in feed:
            rec = item.get('post', {}).get('record', {})
            txt = (rec.get('text') or '').replace('\n', ' ').strip()
            scanned += 1
            hits = pat.findall(txt)
            if hits:
                rows.append({'date': (rec.get('createdAt') or '')[:10], 'handle': h,
                             'keywords': ';'.join(sorted({x.lower() for x in hits})),
                             'text': txt[:280]})
        time.sleep(0.25)
    return pd.DataFrame(rows), scanned

try:
    posts, scanned = scan(accounts)
    if posts.empty: raise RuntimeError('no matches (rate-limited?)')
    print(f'scanned {scanned} posts -> {len(posts)} keyword matches')
except Exception as e:
    p = cache_path('bluesky_health_posts.csv')
    print('Live scan failed:', e, '-> cache', p)
    posts = pd.read_csv(p)
posts.head()


## Step 3: the sanity check that matters (false positives)

Our first version used a naive substring test, `if 'flu' in text.lower()`. It flagged
posts about **"influence"**, because *flu* is a substring of *influence*. The fix is a
word-boundary regex, `\bflu\b`, which is what `scan()` uses above.

This is the whole verify-what-you-scraped habit in one bug: the pipeline ran fine and
returned confident nonsense.

**Prompt used (Lane A):**

> *Show me the difference: count posts matching the substring 'flu' anywhere versus the*
> *word-boundary pattern \bflu\b, and print an example the substring version wrongly*
> *catches.*


In [ ]:
naive = posts['text'].str.contains('flu', case=False, na=False).sum()
strict = posts['keywords'].str.contains(r'\bflu\b', na=False).sum()
print(f"substring 'flu' anywhere : {naive}")
print(f"word-boundary \\bflu\\b     : {strict}")
print('\nexample of what substring matching wrongly catches:')
for t in posts['text'].head(200):
    if 'influen' in str(t).lower() and 'influenza' not in str(t).lower():
        print('  ', str(t)[:100]); break


## Step 4: what is being talked about

**Prompt used (Lane A):**

> *Tally the keywords and plot them, print the five most recent outbreak posts, and save*
> *a tidy CSV.*


In [ ]:
kw = (posts['keywords'].str.split(';').explode().value_counts())
plt.figure(figsize=(9,4))
plt.barh(kw.index[::-1], kw.values[::-1], color='#2A6F97')
plt.xlabel('posts'); plt.title('Disease mentions across public-health accounts (Bluesky)')
plt.tight_layout(); plt.show()
print('most recent outbreak chatter:')
for _, r in posts.sort_values('date', ascending=False).head(5).iterrows():
    print(f"  [{r['date']}] @{str(r['handle'])[:26]:26s} ({r['keywords']}) {str(r['text'])[:80]}")


## Step 5: save, and the honest limits


In [ ]:
posts.to_csv('bluesky_outbreak_chatter.csv', index=False)
print('saved bluesky_outbreak_chatter.csv:', len(posts), 'posts')


## Reflection: limits, and the bigger lesson

- **This is account monitoring, not surveillance.** Bluesky's health community is far
  smaller than Twitter's was, and our watch list is whoever `searchActors` surfaced, so
  it carries real selection bias. Treat it as qualitative early chatter, not a volume
  signal.
- **No auth needed** for discovery and public feeds; full-text search would need a free
  account.
- **The bigger lesson: novel data streams are fragile.** Your own
  `covid_traces_WA.csv` contains Twitter, Kinsa and Cuebiq columns, and *all three of
  those streams are now closed or gone*. What you build on in 2020 can vanish by 2026,
  which is the argument for an agent that can re-scrape whatever exists today.
